In [1]:
import json
from pathlib import Path

def load_json(directory: str) -> list[dict]:
    directory = Path(directory)
    files = [f for f in directory.iterdir() if f.is_file()]
    files.sort()

    data = []
    for file in files:
        with open(file, 'r', encoding='utf-8') as f:
            content = f.read()
            data.append(json.loads(content))

    return data

human_data = load_json("../../datasets/cleaned/human")
ai_data = load_json("../../datasets/cleaned/ai")

print(f"{len(human_data)} human data found!")
print(f"{len(ai_data)} human data found!")

1500 human data found!
1500 human data found!


In [2]:
import pandas as pd

data_sentences = []
for data in human_data + ai_data:
    for sentence in data['sentences']:
        raw_sentence = data['content'][sentence['start']:sentence['end']].strip()
        label = sentence['label']
        lang = data['lang']
        if len(raw_sentence) > 5 and " " in raw_sentence:
            data_sentences.append({
                "lang": lang,
                "sentence": raw_sentence,
                "label": label,
            })

print(f"Extracted total {len(data_sentences)} sentences:")
df = pd.DataFrame(data_sentences)
print(df['label'].value_counts())
print(df['lang'].value_counts())

used_data = 5000
data_per_label = int(used_data / 2)
df = pd.concat([
    df[df['label'] == "human"].iloc[:data_per_label], 
    df[df['label'] == "ai"].iloc[:data_per_label]
])
print(f"\nUsed total {len(df)} sentences:")
print(df['label'].value_counts())
print(df['lang'].value_counts())

Extracted total 7658 sentences:
label
human    3845
ai       3813
Name: count, dtype: int64
lang
id    6223
en    1435
Name: count, dtype: int64

Used total 5000 sentences:
label
human    2500
ai       2500
Name: count, dtype: int64
lang
id    3961
en    1039
Name: count, dtype: int64


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

lang_encoder = LabelEncoder()
label_encoder = LabelEncoder()

x = df[['sentence', 'lang']]
x['lang'] = lang_encoder.fit_transform(x['lang'])
y = label_encoder.fit_transform(df['label'])

x_train, x_test, y_train, y_test = train_test_split(x, y, train_size=0.80, random_state=42)

print(f"Get {len(x_train)} data for training and {len(x_test)} data for testing")

Get 4000 data for training and 1000 data for testing


In [4]:
from transformers import AutoTokenizer

model_name = "microsoft/mdeberta-v3-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def encode_document(item, max_length=128, fixed_size=False):
    text = item['sentence']

    if fixed_size:
        encoding = tokenizer(
            text, 
            truncation=True, 
            max_length=max_length, 
            padding="max_length", 
            return_tensors="pt"
        )
    else:
        encoding = tokenizer(
            text, 
            truncation=True, 
            max_length=max_length, 
            return_offsets_mapping=True, 
            padding=False
        )
        encoding.pop('offset_mapping')
        
    encoding['lang'] = item['lang']

    return encoding

In [5]:
print(x_train.iloc[0])
print(encode_document(x_train.iloc[0]))
print(encode_document(x_train.iloc[0], fixed_size=True))

sentence    Laju pertumbuhan industri otomotif khususnya m...
lang                                                        1
Name: 5572, dtype: object
{'input_ids': [1, 502, 1260, 73479, 38680, 17797, 260, 7167, 46820, 42292, 67757, 22056, 7320, 1784, 304, 44742, 3719, 302, 3665, 693, 25202, 503, 16684, 8720, 458, 51784, 503, 1086, 260, 7994, 514, 260, 7994, 262, 260, 18247, 7523, 60205, 40310, 3654, 260, 14145, 302, 73626, 267, 260, 2146, 72756, 523, 3467, 34675, 260, 10925, 37498, 458, 260, 6886, 694, 50026, 260, 22249, 1405, 6730, 59255, 697, 2], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'lang': n

In [6]:
import torch
import torch.nn as nn
from transformers import AutoModelForSequenceClassification

class SentenceClassifier(nn.Module):
    def __init__(self, model_name=model_name, num_labels=2):
        super().__init__()
        self.encoder = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels, dtype=torch.float32)

        
    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        return outputs.logits

In [7]:
from torch.utils.data import Dataset, DataLoader

class TextDataset(Dataset):
    def __init__(self, data, labels):
        self.data = data
        self.labels = labels

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data.iloc[idx]
        embedding = encode_document(item, fixed_size=True)
        label = self.labels[idx]

        return {
            'sentence': item['sentence'],
            'input_ids': embedding['input_ids'].squeeze(0),
            'attention_mask': embedding['attention_mask'].squeeze(0),
            'lang': item['lang'],
            'label': label,
        }

train_dataset = TextDataset(x_train, y_train)
test_dataset = TextDataset(x_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=True)

In [43]:
from torch.optim import AdamW
from tqdm.auto import tqdm

torch.mps.empty_cache()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = SentenceClassifier().to(device)
optimizer = AdamW(model.parameters(), lr=0.00001)
loss_fn = nn.CrossEntropyLoss()

epochs = 5

for epoch in range(epochs):
    model.train()
    running_test_loss = 0.0
    correct_predictions = 0
    total_samples = 0
    
    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{epochs} [Train]")
    for i, batch in enumerate(train_pbar):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        label = batch['label'].to(device)

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
        label_preds = torch.argmax(logits, dim=1)
        loss = loss_fn(logits, label)
        
        optimizer.zero_grad()
        
        loss.backward()

        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        running_test_loss += loss.item() * len(label)
        correct_predictions += (label_preds == label).sum().item()
        total_samples += len(label)
        
        train_pbar.set_postfix({'loss': f"{loss.item():.4f}"})
            
    final_test_loss = running_test_loss / total_samples
    final_test_acc = correct_predictions / total_samples

    print(f"Training Epoch {epoch + 1} Selesai! [Loss: {final_test_loss:.4f}, Accuracy: {final_test_acc * 100:.2f}%]")

Using device: cpu


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/mdeberta-v3-base
Key                                        | Status     | 
-------------------------------------------+------------+-
lm_predictions.lm_head.dense.weight        | UNEXPECTED | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED | 
mask_predictions.dense.weight              | UNEXPECTED | 
mask_predictions.LayerNorm.bias            | UNEXPECTED | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED | 
mask_predictions.dense.bias                | UNEXPECTED | 
mask_predictions.classifier.bias           | UNEXPECTED | 
lm_predictions.lm_head.bias                | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED | 
mask_predictions.classifier.weight         | UNEXPECTED | 
mask_predictions.LayerNorm.weight          | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
classifier.bias     

Epoch 1/5 [Train]:   0%|          | 0/250 [00:00<?, ?it/s]

Training Epoch 1 Selesai! [Loss: 0.5237, Accuracy: 72.95%]


...

Training Epoch 3 Selesai! [Loss: 0.2389, Accuracy: 91.67%]


Epoch 4/5 [Train]:   0%|          | 0/250 [00:00<?, ?it/s]

Training Epoch 4 Selesai! [Loss: 0.2051, Accuracy: 93.10%]


Epoch 5/5 [Train]:   0%|          | 0/250 [00:00<?, ?it/s]

Training Epoch 5 Selesai! [Loss: 0.1781, Accuracy: 94.15%]


In [44]:
model.eval()
with torch.no_grad():
    running_test_loss = 0.0
    correct_predictions = 0
    total_samples = 0
    
    test_pbar = tqdm(test_loader, desc="[Testing]")
    
    for batch in test_pbar:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        label = batch['label'].to(device)

        preds = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
        label_preds = torch.argmax(preds, dim=1)
        
        loss = loss_fn(preds, label)

        running_test_loss += loss.item() * len(label)
        correct_predictions += (label_preds == label).sum().item()
        total_samples += len(label)

    final_test_loss = running_test_loss / total_samples
    final_test_acc = correct_predictions / total_samples

    print(f"Testing Selesai! [Loss: {final_test_loss:.4f}, Accuracy: {final_test_acc * 100:.2f}%]")

[Testing]:   0%|          | 0/63 [00:00<?, ?it/s]

Testing Selesai! [Loss: 0.4114, Accuracy: 86.40%]


In [47]:
torch.save(model.state_dict(), "../build/ai_detector_5e_1lr.pth")

In [45]:
raw_texts = [
    "Komputer adalah suatu perangkat elektronik yang dapat digunakan untuk mengolah data untuk menghasilkan informasi bagi penggunanya", # human
    "Twitter menjadi media sosial yang populer dan banyak di pakai oleh kalangan milenial ataupun dewasa", # human
    "Sudah menjadi kesadaran bersama bahwa dunia pendidikan merupakan cara yang telah dilakukan umat manusia sepanjang kehidupannya untuk menjadi sarana dalam melakukan transmisi dan transformasi baik nilai maupun ilmu pengetahuan", #human
    "Sampah adalah adalah sisa atau barang buangan yang sudah tidak digunakan dan dipakai lagi oleh pemiliknya", # human
    "Kembali lagi ke drama penculikan Soekarno-Hatta, dengan menggunakan sebuah mobil, Soekarni, Wikana, Aidit, dan Chaerul Saleh dari perkumpulan Menteng 31, mengangkut Soekarno dan Hatta ke Kota Rengasdengklok", # human
    "Secara umum, komputer adalah perangkat elektronik yang digunakan untuk menerima, mengolah, dan menyimpan data berdasarkan instruksi atau program tertentu sehingga dapat menghasilkan informasi yang berguna bagi pengguna", # ai
    "Sampah adalah barang atau sisa dari suatu kegiatan yang sudah tidak digunakan lagi dan dianggap tidak memiliki manfaat oleh pemiliknya", # ai
    "Dunia pendidikan telah menjadi bagian penting dalam kehidupan manusia karena berperan sebagai sarana untuk mewariskan sekaligus mengembangkan nilai-nilai dan ilmu pengetahuan dari satu generasi ke generasi berikutnya", # ai
    "Kemerdekaan Indonesia diproklamasikan pada tanggal 17 Agustus 1945 di Jalan Pegangsaan Timur No. 56, Jakarta Pusat", # ai
    "Real Madrid adalah klub sepak bola profesional asal Madrid, Spanyol, yang berkompetisi di La Liga dan merupakan salah satu klub tersukses di dunia" # ai
]

encoding_raw_texts = [encode_document({'sentence': text, 'lang': 0}, fixed_size=True) for text in raw_texts]

In [46]:
print(f"Label encoder: {label_encoder.classes_}")
classes = label_encoder.classes_

model.eval()
with torch.no_grad():
    for item in encoding_raw_texts:
        input_ids = item['input_ids'].to(device)
        attention_mask = item['attention_mask'].to(device)

        preds = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = nn.functional.softmax(preds, dim=1)
        label_preds = torch.argmax(preds, dim=1)
        confidence = torch.round(probs * 100, decimals=2)
        print(f"- Predicted as {classes[label_preds.squeeze(0)]} with confidence {torch.max(confidence)}%")

Label encoder: ['ai' 'human']
- Predicted as human with confidence 99.5999984741211%
- Predicted as human with confidence 99.77999877929688%
- Predicted as human with confidence 94.29000091552734%
- Predicted as human with confidence 99.66000366210938%
- Predicted as human with confidence 95.72000122070312%
- Predicted as ai with confidence 97.51000213623047%
- Predicted as human with confidence 92.44000244140625%
- Predicted as ai with confidence 90.0199966430664%
- Predicted as ai with confidence 98.1500015258789%
- Predicted as ai with confidence 96.79000091552734%
